In [2]:
```python
# ============================================================
# REAL ESTATE VALUATION DATASET
# K-MEANS CLUSTERING + NLP
# COMPLETE JUPYTER NOTEBOOK CODE
# ============================================================


# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import os
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.sparse import csr_matrix, hstack

from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

warnings.filterwarnings("ignore")

print("Libraries imported successfully.")


# ============================================================
# 2. LOAD DATASET
# ============================================================

file_path = "Real estate valuation data set.csv"

if not os.path.exists(file_path):
    raise FileNotFoundError(
        f"File '{file_path}' was not found. "
        "Make sure the CSV file is in the same folder as this notebook."
    )

df = pd.read_csv(file_path)

print("Dataset loaded successfully.")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

display(df.head())


# ============================================================
# 3. CLEAN COLUMN NAMES
# ============================================================

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

print("Cleaned column names:")
print(df.columns.tolist())


# ============================================================
# 4. BASIC DATASET INFORMATION
# ============================================================

print("\nDataset shape:")
print(df.shape)

print("\nDataset information:")
df.info()

print("\nMissing values:")
display(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())


# ============================================================
# 5. REMOVE DUPLICATE ROWS
# ============================================================

df = df.drop_duplicates().reset_index(drop=True)

print("Shape after removing duplicates:")
print(df.shape)


# ============================================================
# 6. HANDLE MISSING VALUES
# ============================================================

numeric_columns = df.select_dtypes(
    include=np.number
).columns.tolist()

text_columns = df.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

# Numeric missing values -> median
for column in numeric_columns:
    df[column] = df[column].fillna(
        df[column].median()
    )

# Text missing values -> empty string
for column in text_columns:
    df[column] = df[column].fillna("")

print("Missing values after cleaning:")
display(df.isnull().sum())


# ============================================================
# 7. DESCRIPTIVE STATISTICS
# ============================================================

print("Descriptive statistics:")

display(
    df.describe(include="all").T
)


# ============================================================
# 8. IDENTIFY THE TARGET/PRICE COLUMN
# ============================================================

# Standard dataset usually contains:
# house_price_of_unit_area

possible_price_columns = [
    "house_price_of_unit_area",
    "house_price_per_unit_area",
    "house_price",
    "property_price",
    "price"
]

price_column = None

for column in possible_price_columns:
    if column in df.columns:
        price_column = column
        break

print("Detected price column:", price_column)


# ============================================================
# 9. NUMERICAL FEATURES FOR K-MEANS
# ============================================================

numeric_columns = df.select_dtypes(
    include=np.number
).columns.tolist()

# Do not use the target/property price to form clusters
features_numeric = [
    column
    for column in numeric_columns
    if column != price_column
]

print("\nNumerical features used for clustering:")
for column in features_numeric:
    print("-", column)


# ============================================================
# 10. CHECK FOR CONSTANT COLUMNS
# ============================================================

constant_columns = [
    column
    for column in features_numeric
    if df[column].nunique() <= 1
]

if constant_columns:
    print("\nRemoving constant columns:")
    print(constant_columns)

    features_numeric = [
        column
        for column in features_numeric
        if column not in constant_columns
    ]


# ============================================================
# 11. STANDARDIZE NUMERICAL FEATURES
# ============================================================

X_numeric = df[features_numeric].copy()

scaler = StandardScaler()

X_numeric_scaled = scaler.fit_transform(
    X_numeric
)

print(
    "\nScaled numerical feature matrix:",
    X_numeric_scaled.shape
)


# ============================================================
# 12. NLP SECTION
# ============================================================

print("\nText columns detected:")
print(text_columns)

# Exclude columns that are not useful as NLP text
usable_text_columns = []

for column in text_columns:

    # Only use a column if it contains actual non-empty text
    non_empty = (
        df[column]
        .astype(str)
        .str.strip()
        .replace("nan", "")
    )

    if non_empty.str.len().sum() > 0:
        usable_text_columns.append(column)

print("\nUsable NLP columns:")
print(usable_text_columns)


# ============================================================
# 13. CREATE COMBINED TEXT
# ============================================================

if len(usable_text_columns) > 0:

    df["combined_text"] = (
        df[usable_text_columns]
        .astype(str)
        .agg(" ".join, axis=1)
        .str.lower()
    )

    # Basic NLP cleaning
    df["combined_text"] = (
        df["combined_text"]
        .str.replace(r"[^a-zA-Z0-9\s]", " ", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    print("\nExample NLP data:")
    display(
        df[["combined_text"]].head()
    )

else:

    df["combined_text"] = ""

    print(
        "\nNo text columns were found."
    )

    print(
        "The standard Real Estate Valuation dataset "
        "is numerical, so NLP will not be used."
    )


# ============================================================
# 14. TF-IDF TRANSFORMATION
# ============================================================

use_nlp = False
X_text = None

if len(usable_text_columns) > 0:

    valid_text = (
        df["combined_text"]
        .str.strip()
        .str.len()
        .sum()
    )

    if valid_text > 0:

        tfidf = TfidfVectorizer(
            lowercase=True,
            stop_words="english",
            max_features=100,
            ngram_range=(1, 2)
        )

        X_text = tfidf.fit_transform(
            df["combined_text"]
        )

        use_nlp = True

        print(
            "TF-IDF transformation completed."
        )

        print(
            "TF-IDF matrix shape:",
            X_text.shape
        )

        print(
            "\nTop TF-IDF terms:"
        )

        terms = tfidf.get_feature_names_out()

        print(
            terms[:30]
        )


# ============================================================
# 15. COMBINE NUMERICAL + NLP FEATURES
# ============================================================

X_numeric_sparse = csr_matrix(
    X_numeric_scaled
)

if use_nlp:

    X = hstack(
        [
            X_numeric_sparse,
            X_text
        ]
    ).tocsr()

    print(
        "Numerical + NLP features combined."
    )

else:

    X = X_numeric_sparse

    print(
        "Only numerical features are used."
    )

print(
    "Final feature matrix:",
    X.shape
)


# ============================================================
# 16. ELBOW METHOD
# ============================================================

inertia = []

k_values = range(2, 11)

for k in k_values:

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20
    )

    model.fit(X)

    inertia.append(
        model.inertia_
    )


plt.figure(figsize=(10, 6))

plt.plot(
    list(k_values),
    inertia,
    marker="o"
)

plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method for K-Means")

plt.xticks(
    list(k_values)
)

plt.grid(True)

plt.tight_layout()
plt.show()


# ============================================================
# 17. SILHOUETTE ANALYSIS
# ============================================================

silhouette_scores = []

for k in k_values:

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20
    )

    labels = model.fit_predict(X)

    score = silhouette_score(
        X,
        labels
    )

    silhouette_scores.append(
        score
    )

    print(
        f"K = {k}, "
        f"Silhouette Score = {score:.4f}"
    )


plt.figure(figsize=(10, 6))

plt.plot(
    list(k_values),
    silhouette_scores,
    marker="o"
)

plt.xlabel("Number of Clusters (K)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score for Different K Values")

plt.xticks(
    list(k_values)
)

plt.grid(True)

plt.tight_layout()
plt.show()


# ============================================================
# 18. SELECT BEST K
# ============================================================

best_index = np.argmax(
    silhouette_scores
)

best_k = list(k_values)[
    best_index
]

best_score = silhouette_scores[
    best_index
]

print(
    "\nBest K:",
    best_k
)

print(
    "Best silhouette score:",
    round(best_score, 4)
)


# ============================================================
# 19. FINAL K-MEANS MODEL
# ============================================================

kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=20
)

df["cluster"] = kmeans.fit_predict(
    X
)

print(
    "\nK-Means clustering completed."
)


# ============================================================
# 20. CLUSTER COUNTS
# ============================================================

cluster_counts = (
    df["cluster"]
    .value_counts()
    .sort_index()
)

print(
    "Number of properties in each cluster:"
)

display(
    cluster_counts.to_frame(
        name="Number_of_Properties"
    )
)


# ============================================================
# 21. CLUSTER PERCENTAGES
# ============================================================

cluster_percentages = (
    df["cluster"]
    .value_counts(
        normalize=True
    )
    .sort_index()
    * 100
)

cluster_percentage_table = (
    cluster_percentages
    .round(2)
    .to_frame(
        name="Percentage"
    )
)

display(
    cluster_percentage_table
)


# ============================================================
# 22. NUMERICAL CLUSTER SUMMARY
# ============================================================

cluster_summary = (
    df.groupby("cluster")[features_numeric]
    .mean()
    .round(3)
)

print(
    "Average numerical features by cluster:"
)

display(
    cluster_summary
)


# ============================================================
# 23. PRICE SUMMARY BY CLUSTER
# ============================================================

if price_column is not None:

    price_summary = (
        df.groupby("cluster")[price_column]
        .agg(
            Count="count",
            Mean="mean",
            Median="median",
            Minimum="min",
            Maximum="max"
        )
        .round(3)
    )

    print(
        "Property price summary by cluster:"
    )

    display(
        price_summary
    )


# ============================================================
# 24. VISUALIZE CLUSTER SIZE
# ============================================================

plt.figure(figsize=(8, 5))

sns.barplot(
    x=cluster_counts.index,
    y=cluster_counts.values
)

plt.xlabel("Cluster")
plt.ylabel("Number of Properties")
plt.title("Number of Properties in Each Cluster")

plt.tight_layout()
plt.show()


# ============================================================
# 25. PCA
# ============================================================

# Convert final matrix to dense format
X_dense = X.toarray()

pca = PCA(
    n_components=2,
    random_state=42
)

X_pca = pca.fit_transform(
    X_dense
)

df["PCA1"] = X_pca[:, 0]
df["PCA2"] = X_pca[:, 1]

explained_variance = (
    pca.explained_variance_ratio_
)

print(
    "PCA explained variance:"
)

print(
    explained_variance
)

print(
    "Total explained variance:",
    round(
        explained_variance.sum(),
        4
    )
)


# ============================================================
# 26. PCA CLUSTER VISUALIZATION
# ============================================================

plt.figure(figsize=(11, 7))

sns.scatterplot(
    data=df,
    x="PCA1",
    y="PCA2",
    hue="cluster",
    palette="Set1",
    s=80,
    alpha=0.8
)

plt.title(
    "K-Means Clustering of Real Estate Properties"
)

plt.xlabel(
    "Principal Component 1"
)

plt.ylabel(
    "Principal Component 2"
)

plt.legend(
    title="Cluster"
)

plt.grid(True)

plt.tight_layout()
plt.show()


# ============================================================
# 27. PRICE DISTRIBUTION BY CLUSTER
# ============================================================

if price_column is not None:

    plt.figure(figsize=(10, 6))

    sns.boxplot(
        data=df,
        x="cluster",
        y=price_column
    )

    plt.title(
        "House Price Distribution by Cluster"
    )

    plt.xlabel("Cluster")
    plt.ylabel("House Price per Unit Area")

    plt.grid(
        axis="y",
        alpha=0.3
    )

    plt.tight_layout()
    plt.show()


# ============================================================
# 28. FEATURE DISTRIBUTION BY CLUSTER
# ============================================================

for feature in features_numeric:

    plt.figure(figsize=(9, 5))

    sns.boxplot(
        data=df,
        x="cluster",
        y=feature
    )

    plt.title(
        f"{feature} by Cluster"
    )

    plt.xlabel("Cluster")
    plt.ylabel(feature)

    plt.tight_layout()
    plt.show()


# ============================================================
# 29. FIND REPRESENTATIVE PROPERTIES
# ============================================================

# Distance from each observation to each cluster center
distances = kmeans.transform(X)

# Distance to assigned cluster
df["cluster_distance"] = [
    distances[i, cluster]
    for i, cluster in enumerate(
        df["cluster"]
    )
]

print(
    "Representative properties:"
)

representative_properties = (
    df.sort_values(
        ["cluster", "cluster_distance"]
    )
    .groupby("cluster")
    .head(5)
)

display(
    representative_properties
)


# ============================================================
# 30. DISPLAY PROPERTIES IN EACH CLUSTER
# ============================================================

for cluster_number in sorted(
    df["cluster"].unique()
):

    print("\n")
    print("=" * 70)
    print(
        f"CLUSTER {cluster_number}"
    )
    print("=" * 70)

    cluster_data = df[
        df["cluster"] == cluster_number
    ]

    print(
        "Number of properties:",
        len(cluster_data)
    )

    display(
        cluster_data.head(10)
    )


# ============================================================
# 31. CLUSTER CHARACTERISTICS
# ============================================================

print(
    "\nCLUSTER CHARACTERISTICS"
)

for cluster_number in sorted(
    df["cluster"].unique()
):

    cluster_data = df[
        df["cluster"] == cluster_number
    ]

    print("\n" + "=" * 60)
    print(
        f"Cluster {cluster_number}"
    )
    print("=" * 60)

    for feature in features_numeric:

        mean_value = (
            cluster_data[feature]
            .mean()
        )

        print(
            f"{feature}: {mean_value:.3f}"
        )

    if price_column is not None:

        average_price = (
            cluster_data[price_column]
            .mean()
        )

        print(
            f"{price_column}: "
            f"{average_price:.3f}"
        )


# ============================================================
# 32. OVERALL MODEL EVALUATION
# ============================================================

final_silhouette = silhouette_score(
    X,
    df["cluster"]
)

print("\n")
print("=" * 70)
print("MODEL EVALUATION")
print("=" * 70)

print(
    "Algorithm: K-Means"
)

print(
    "Number of clusters:",
    best_k
)

print(
    "Silhouette score:",
    round(
        final_silhouette,
        4
    )
)

print(
    "Inertia:",
    round(
        kmeans.inertia_,
        4
    )
)


# ============================================================
# 33. SAVE CLUSTERED DATASET
# ============================================================

output_file = (
    "Real_Estate_Valuation_KMeans_Results.csv"
)

df.to_csv(
    output_file,
    index=False
)

print(
    "\nResults saved to:"
)

print(
    output_file
)


# ============================================================
# 34. SAVE CLUSTER SUMMARY
# ============================================================

summary_file = (
    "Real_Estate_Cluster_Summary.csv"
)

cluster_summary.to_csv(
    summary_file
)

print(
    "Cluster summary saved to:"
)

print(
    summary_file
)


# ============================================================
# 35. FINAL RESULT
# ============================================================

print("\n")
print("=" * 70)
print("FINAL RESULT")
print("=" * 70)

print(
    f"Dataset rows: {len(df)}"
)

print(
    f"Features used: {len(features_numeric)}"
)

print(
    f"Best K: {best_k}"
)

print(
    f"Silhouette Score: {final_silhouette:.4f}"
)

print(
    f"NLP used: {use_nlp}"
)

print(
    "\nCluster sizes:"
)

display(
    df["cluster"]
    .value_counts()
    .sort_index()
    .to_frame(
        "Number of Properties"
    )
)

print(
    "\nNotebook completed successfully."
)
```


SyntaxError: invalid syntax (1383844864.py, line 1)